# 弱監督與語音評估

[完整講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/e2-weak-supervision-wer.md) · [在 Colab 執行](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/e2_weak_supervision_wer.ipynb)

本練習使用自編小型資料，在 CPU 執行。先手算替換、刪除和插入，再按組比較模型與自動標籤的 WER。

Radford 等人的 Whisper 論文問的是：**用大量來自網路的弱標註語音訓練後，模型在未針對測試資料集微調時，能否辨識更多樣的真實語音？** 本課先讀 §2.1 的資料處理，再用 §3.2.1 Figure 2 和 Table 2 練習解讀零樣本穩健性。

## 論文如何使用弱監督？

| 階段 | 論文用的做法 | 讀者要分辨的事 |
|---|---|---|
| 訓練資料（§2.1） | 蒐集網路上配有文字的約 68 萬小時多語、多任務音訊，清理音文配對，切成約 30 秒片段 | 弱標註來自現有文字或字幕；它可以擴大資料覆蓋，也可能和語音內容對不上 |
| 訓練任務（§2.3） | 在同一模型用任務與語言 token 指定轉錄、翻譯等任務 | 比較 WER 前，先說清楚是在做原語言轉錄還是翻譯 |
| 零樣本評估（§3.2.1） | 用現成模型到各語音資料集評估，套用論文的文字正規化後計算 WER | 除了看單一基準分數，還要看分布改變後錯誤增加多少 |

## Table 2 的比較設計

作者以 LibriSpeech test-clean 表現相近的兩個模型做對照，接著比較其他英語資料集。以下是 Table 2 的部分數字；WER 越低越好。

| 資料集 | wav2vec 2.0 Large（無 LM）WER | Whisper Large V2 WER | 相對錯誤減少（RER） |
|---|---:|---:|---:|
| LibriSpeech Clean（參考組） | 2.7% | 2.7% | 0.0% |
| Common Voice | 29.9% | 9.0% | 69.9% |
| Switchboard | 28.3% | 13.8% | 51.2% |
| LibriSpeech Other | 6.2% | 5.2% | 16.1% |

以 Common Voice 為例，WER 下降 $29.9-9.0=20.9$ 個百分點。把下降幅度除以原本的 29.9，才得到相對錯誤減少約 69.9%。Notebook 會依原表其他 13 個資料集的結果重算平均相對錯誤減少，約 55.2%。計算時**先逐資料集算相對下降，再平均**；若改用兩個平均 WER 相除，算的是另一個量。

這組比較顯示在參考集表現相近時，模型在其他來自不同語域或錄音條件的資料集上仍可能有明顯差距。兩個系統在訓練資料、方法和架構上有多項差異，因此這張表支持的是跨資料集表現的比較。若要單獨估計弱標註的影響，需另外控制訓練條件。[Whisper 原論文](https://arxiv.org/abs/2212.04356)

## 合成暖身與手算

## 一句話要改幾個詞？

字詞錯誤率（WER）先將模型輸出和參考文字逐詞對齊，再找最少要做多少次替換 $S$、刪除 $D$、插入 $I$。分母是參考文字的詞數 $N$：

$$\mathrm{WER}=\frac{S+D+I}{N}.$$

例如參考是 `please open the door`，模型輸出 `please close door`。把 `open` 換成 `close`，刪掉 `the`，需要 2 次編輯，WER 為 $2/4=50\%$。參考只有一詞而輸出多出兩詞時，插入數可以超過分母，因此 WER 可能高於 100%。中文等情境要先定義分詞口徑，也可以另報字元錯誤率（CER）。

## 先按組算，再解讀整體 WER

以下是自編範例。A、B 可代表不同錄音條件或不同口音組，參考文字都由教師提供。

| 群組 | 錯誤數 | 參考詞數 | WER |
|---|---:|---:|---:|
| A | 1 | 8 | 12.5% |
| B | 4 | 8 | 50% |
| 合計 | 5 | 16 | 31.25% |

整體 WER 用全部錯誤數除以全部參考詞數。這個例子給兩組相同詞數；若詞數不同，整體分數會較受長音訊群組影響。Notebook 另外算自動標籤相對人工參考的 WER，示範如何檢查訓練標籤與最終評估的品質。群組差距也可能來自麥克風、噪聲或文本難度，要先控制或分層再談原因。

## 執行環境

Colab 通常已有 NumPy 和 Matplotlib。若要使用本專案規定範圍內的套件，可執行下一格。

In [ ]:
%pip -q install 'numpy>=1.26,<3' 'matplotlib>=3.8,<4'

## 程式驗證

先預測結果，再執行下方的手算核對和視覺化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def edit_counts(reference, prediction):
    ref, pred = reference.split(), prediction.split()
    edits = [[0] * (len(pred) + 1) for _ in range(len(ref) + 1)]
    move = [[None] * (len(pred) + 1) for _ in range(len(ref) + 1)]
    for i in range(1, len(ref) + 1):
        edits[i][0], move[i][0] = i, "D"
    for j in range(1, len(pred) + 1):
        edits[0][j], move[0][j] = j, "I"
    for i, reference_word in enumerate(ref, 1):
        for j, predicted_word in enumerate(pred, 1):
            mismatch = reference_word != predicted_word
            candidates = (
                (edits[i - 1][j - 1] + mismatch, "S" if mismatch else "match"),
                (edits[i - 1][j] + 1, "D"),
                (edits[i][j - 1] + 1, "I"),
            )
            edits[i][j], move[i][j] = min(candidates, key=lambda option: option[0])

    i, j = len(ref), len(pred)
    errors = {"S": 0, "D": 0, "I": 0}
    while i or j:
        action = move[i][j]
        if action in errors:
            errors[action] += 1
        if action in ("match", "S", "D"):
            i -= 1
        if action in ("match", "S", "I"):
            j -= 1
    return errors, len(ref)

def corpus_wer(pairs):
    totals = [edit_counts(reference, output) for reference, output in pairs]
    return sum(sum(errors.values()) for errors, _ in totals) / sum(n for _, n in totals)

data = {
    "A": [
        ("open the door now", "open the door now", "open the door"),
        ("please close the door", "please close door", "please close the door"),
    ],
    "B": [
        ("open the window now", "open the door right now", "open the door"),
        ("please close the window", "close window", "close door"),
    ],
}
assert edit_counts("please open the door", "please close door")[0] == {"S": 1, "D": 1, "I": 0}
assert corpus_wer([("hello", "hello there again")]) == 2
wer_actual = [corpus_wer([(ref, hyp) for ref, hyp, _ in items]) for items in data.values()]
wer_labels = [corpus_wer([(ref, label) for ref, _, label in items]) for items in data.values()]
overall = corpus_wer([(ref, hyp) for items in data.values() for ref, hyp, _ in items])
np.testing.assert_allclose(wer_actual, [0.125, 0.5])
np.testing.assert_allclose(overall, 0.3125)
print("Model WER by group:", wer_actual, "; pooled:", overall)
print("Automatic-label WER by group:", wer_labels)

positions = np.arange(len(data))
fig, ax = plt.subplots()
ax.bar(positions - 0.18, wer_labels, width=0.35, label="Automatic labels")
ax.bar(positions + 0.18, wer_actual, width=0.35, label="Model output")
ax.set(xticks=positions, xticklabels=data, ylabel="WER", title="Audit each group separately")
ax.legend()
plt.show()

## 原文對照｜重算 Whisper Table 2 的相對錯誤減少

依上方的原文段落先圈出資料集、指標或公式，再用程式核對。圖表依旁邊標註的來源解讀。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Radford et al., Table 2. Values are WER percentages as printed in the paper.
reference = ("LibriSpeech Clean", 2.7, 2.7)
rows = [
    ("Artie", 24.5, 6.2), ("Common Voice", 29.9, 9.0), ("Fleurs En", 14.6, 4.4),
    ("Tedlium", 10.5, 4.0), ("CHiME6", 65.8, 25.5), ("VoxPopuli En", 17.9, 7.3),
    ("CORAAL", 35.6, 16.2), ("AMI IHM", 37.0, 16.9), ("Switchboard", 28.3, 13.8),
    ("CallHome", 34.8, 17.6), ("WSJ", 7.7, 3.9), ("AMI SDM1", 67.6, 36.4),
    ("LibriSpeech Other", 6.2, 5.2),
]
baseline, whisper = np.array([(old, new) for _, old, new in rows]).T
relative_each = (baseline - whisper) / baseline
macro_relative = relative_each.mean()
ratio_of_means = (baseline.mean() - whisper.mean()) / baseline.mean()
assert round(100 * macro_relative, 1) == 55.2
assert round(100 * relative_each[1], 1) == 69.9
print(f"Average of per-dataset relative reductions: {macro_relative:.1%}")
print(f"Relative reduction of the two average WERs: {ratio_of_means:.1%}")
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(baseline, whisper, label="Other datasets in Table 2")
ax.scatter(reference[1], reference[2], marker="s", label=reference[0])
ax.plot([0, 70], [0, 70], "--", color="gray", label="Equal WER")
for name in ("Common Voice", "Switchboard", "LibriSpeech Other"):
    _, x, y = next(row for row in rows if row[0] == name)
    ax.annotate(name, (x, y), xytext=(5, 4), textcoords="offset points", fontsize=8)
ax.set(xlabel="wav2vec 2.0 Large (no LM) WER (%)", ylabel="Whisper Large V2 WER (%)",
       title="Selected comparison from Whisper Table 2")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 自我檢核與練習

**對照原文。** 用 Table 2 的 Switchboard 算一次「百分點」與「相對錯誤減少」。參考答案是 14.5 個百分點與約 51.2%。說明為什麼 Figure 2 用 LibriSpeech 作為參考，以及你的自編口音分組和原表跨資料集比較各能排查哪一類問題。

把其中一組加入一筆長句，重算各組和整體 WER。解釋為什麼整體 WER 要加總錯誤與參考詞數，以及群組差距還需要哪些資料才能解釋。

## 來源

- [Whisper 原論文](https://arxiv.org/abs/2212.04356)
- [NIST 語音評估工具](https://www.nist.gov/itl/iad/mig/tools)